# CYGNSS L1 operator story figure: one observation, native pixels to tile update

Methods schematic for the CYGNSS L1 paper. The figure follows one assimilated L1 observation from the native-resolution
IGOT pixels (Copernicus DEM, about 30 m) through aggregation to M36 tile coefficients, and on to the analysis increment
it produces in the fixed-operator runs.

**Observation.** cyg02, sample 163383, ch 3, 2020-11-15 22:41 UTC (the 2020-11-16 00z window), SW Arizona. It was chosen on
Discover (`docs/discover_operator_story_figure_handoff.md`; candidate table in the bundle's `selection/`) because it is
nearly isolated (7 L1 obs within ±1°), has good SNR (10.6 dB) and low incidence (17.2°), and its owner tile carries 80% of
the coefficient weight.

**Operator.** The production product uses `vegetation_mode = tile-opacity-sp`. For the 3×5 crop-sum observable,

$$H(x) = s\sum_{p} K_p\,\sigma_p\,A_{t(p)}\,R_{t(p)}(x) = \sum_t C_t R_t(x), \qquad C_t = s\sum_{p\in t} K_p\,\sigma_p\,A_t$$

- $K_p$: unnormalized WAF/area kernel of the selected DDM bins
- $\sigma_p = \sigma_{0,\mathrm{base}}/R_\mathrm{base}$: the fixed IGOT pixel factor (terrain, roughness, geometry)
- $A_t = \exp(-\tau_t \cdot 2\sec\theta_\mathrm{sp})$: tile vegetation attenuation
- $R_t$: tile LR reflectivity from the GEOSldas SFMC forecast
- $s$: the reflectivity-domain scale

The notebook recomputes $C_t$ from the pixels and checks it against the product, so the pixel panels are the operator
that was actually assimilated.

**Panels**

| Row | Content |
|---|---|
| 1. Observation | (a) L1 DDM with 3×5 crop · (b) Copernicus DEM · (c) crop WAF footprint $K_p$ |
| 2. Pixel physics | (d) IGOT pixel factor $\sigma_p$ · (e) vegetation attenuation $A_t$ · (f) forecast reflectivity $R_t$ |
| 3. Pixels to tiles | (g) pixel contribution to $H(x)$ · (h) tile coefficient $C_t$ · (i) tile share $C_tR_t/H$ |
| 4. DA update | (j) forecast SFMC · (k) SRFEXC increment · (l) SFMC analysis − forecast |

Map panels share the DEM extent and show M36 tile outlines. Tile fields are painted on the pixel grid through the
pixel-to-tile mapping, so tile boundaries are identical in every panel.

**Inputs.**
- The Discover bundle, copied to `CYGNSS_operator/artifacts/cygl1_story_figure_bundle/`.
- The raw L1 file for 2020-11-15 cyg02, from `scripts/download_cygnss_l1_range.py 2020-11-15 --spacecraft 2`.
- The local Copernicus DEM tiles.

**Kernel.** Needs the `cygnss` env (IGOT, GDAL): `/Users/amfox/mamba/envs/cygnss/bin/python`. `regrid` has no `osgeo`.
The IGOT setup cell takes about a minute.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.ticker import MultipleLocator

# ===============================
# CONFIG
# ===============================
OP_ROOT = Path("/Users/amfox/Desktop/CYGNSS_operator")
BUNDLE = OP_ROOT / "artifacts/cygl1_story_figure_bundle"
GA_PROJ = Path("/Users/amfox/Desktop/geosldas-analysis/projects/CYGNSS_L1_AZ")
OUT_DIR = GA_PROJ / "output/operator_story_20201116_0000z"

SAMPLE_ID, CH_ID = 163383, 3
CYCLE = "20201116_0000z"
IGOT_CONFIG = BUNDLE / "igot_config/20201115_doy320_cyg02_selected44/run_config.ini"
PRODUCT = BUNDLE / "shared/cygnss_l1_ddm3x5_crop_scalar_m36_20201116_all_cyg.nc4"
MWRTM = BUNDLE / "shared/DA_L1_full_xc015_coh040216_err39_fixedop.ldas_mwRTMparam.20201101_0000z.nc4"
VEGOPACITY = BUNDLE / "shared/vegopacity.bin.domain"
TILE_INDEX_REF = BUNDLE / "shared/LS_DAv8_M36_as_20191101.inst3_1d_lndfcstana_Nt.20191101_0300z.nc4"
COPDEM_ROOT = Path("/Users/amfox/Desktop/cygnss_data/copernicus_dem")
TEMP_DEM_ROOT = Path("/private/tmp/cygl1_operator_story_dem")

# The two candidate primary L1 arms (the choice is still open); both assimilate the same obs.
ARMS = {
    "err39": "DA_L1_full_xc015_coh040216_err39_fixedop",
    "err275": "DA_L1_full_xc015_coh040216_fixedop",
}

# IGOT and the story script resolve some defaults relative to the operator repo.
os.chdir(OP_ROOT)
sys.path.insert(0, str(OP_ROOT / "scripts"))
import plot_cygnss_good_obs_operator_story as story  # data loaders shared with the script

OUT_DIR.mkdir(parents=True, exist_ok=True)
for path in [IGOT_CONFIG, PRODUCT, MWRTM, VEGOPACITY, TILE_INDEX_REF]:
    assert path.exists(), path

## IGOT observation setup (arm-independent)

This builds the IGOT operator components for the observation: the observed DDM, WAF cache, pixel σ₀ and the DEM crop.
`COPERNICUS_DEM_PATH` must be set before the IGOT modules are imported, because IGOT reads it at import time.

In [ ]:
import copy

story.prepare_temp_copdem_root(TEMP_DEM_ROOT, COPDEM_ROOT)

from util.run_helper import import_igot_config_file
from cygnss_operator.case_builder import build_cygnss_operator_components, observable_spec_from_cli
from cygnss_operator.geosldas_bridge import load_geosldas_tile_data, tile_lr_reflectivity
from cygnss_operator.operator_api import compute_waf_kernel
from cygnss_operator.pixel_ensemble_benchmark import map_pixels_to_grid_index_tiles, pixel_lon_lat_grid
from cygnss_operator.scalar_observables import crop_slices, linear_to_db, resolve_center_index
from cygnss_operator.vegetation_opacity import read_vegopacity_for_date, two_path_sp_incidence_geometry_factor

# story.build_tile_values uses this name as a module global (normally set inside story.main()).
story.tile_lr_reflectivity = tile_lr_reflectivity

config = copy.deepcopy(import_igot_config_file(str(IGOT_CONFIG)))
config.update(sample_id=SAMPLE_ID, ch_id=CH_ID, plot_fig=False, save_fig=False)
components = build_cygnss_operator_components(
    config, domain="reflectivity", fig_out_folder=None,
    download_cygnss=False, download_dem=True, verbose=False,
    compute_pixel_vegetation_geometry=True,
)

meta = components.metadata["cygnss"]
dem_extent = tuple(float(x) for x in components.metadata["dem"]["dem_img_ext"])
dem = story.read_dem_band(story.find_matching_dem(TEMP_DEM_ROOT / "cropped", dem_extent), 1)

observable = observable_spec_from_cli("crop3x5", center="reference_peak")
center = resolve_center_index(components.observed_ddm, observable.center, reference_ddm=components.observed_ddm)
crop = crop_slices(components.observed_ddm.shape, center, observable.crop_shape)
K = compute_waf_kernel(components.cache, center, observable.crop_shape, normalize=False)
sigma_factor = np.where(components.sigma0 > 0.0, components.sigma0 / components.base_lr_reflectivity, 0.0)
domain_scale = float(components.cache.reflectivity_scale)
geometry_factor = two_path_sp_incidence_geometry_factor(float(components.incidence_angle_deg))

pix_shape = K.shape
pix_lon, pix_lat = pixel_lon_lat_grid(dem_extent, pix_shape)
dx_km = (dem_extent[1] - dem_extent[0]) / pix_shape[1] * 111.32 * np.cos(np.deg2rad(meta["sp_lat"]))
dy_km = (dem_extent[3] - dem_extent[2]) / pix_shape[0] * 111.32
print(f"obs: cyg{int(meta['sc_num']):02d} sample {meta['sample_id']} ch {meta['ch_id']}  {meta['ddm_time']}")
print(f"sp: {meta['sp_lon']:.4f}, {meta['sp_lat']:.4f}  incidence {components.incidence_angle_deg:.2f} deg  "
      f"2 sec(theta) = {geometry_factor:.3f}")
print(f"pixel grid: {pix_shape}  ~{dx_km*1e3:.0f} m x {dy_km*1e3:.0f} m  extent {dem_extent}")
print(f"active kernel pixels: {np.count_nonzero(K > 0):,}")

## Per-arm fields and pixel-to-tile aggregation

For each arm, this loads the GEOSldas state and increments, maps DEM pixels onto that arm's tiles, and recomputes
$C_t$ and $H(x)$ from the pixels.

The product's `tile_index0` uses the global 2019 reference numbering. `story.remap_product_tile_indices` moves it onto
the arm's 909-tile domain ordering via IG/JG.

In [ ]:
def paint(tile_values, mapping):
    '''Tile field -> pixel grid via the pixel-to-tile mapping (NaN where unmapped or NaN).'''
    out = np.full(mapping.shape, np.nan)
    ok = mapping >= 0
    out[ok] = np.asarray(tile_values, dtype=float)[mapping[ok]]
    return out


def arm_fields(arm):
    exp = ARMS[arm]
    inst3 = BUNDLE / arm / f"{exp}.inst3_1d_lndfcstana_Nt.{CYCLE}.nc4"
    incr = BUNDLE / arm / f"{exp}.catch_progn_incr.{CYCLE}.nc4"
    tiles = load_geosldas_tile_data(inst3, MWRTM)
    ntile = tiles.tile_index0.size
    tau, _ = read_vegopacity_for_date(VEGOPACITY, meta["ddm_time"].date(), ntile)
    product = story.read_product_observation(PRODUCT, SAMPLE_ID, CH_ID)
    story.remap_product_tile_indices(product, tiles, TILE_INDEX_REF)
    increments = story.read_increment_fields(incr)
    tv = story.build_tile_values(tiles, product, increments, tau, components)

    mapping = map_pixels_to_grid_index_tiles(dem_extent, pix_shape, tiles).pixel_tile_indices0

    # Tile quantities for every tile (not just support tiles), so they can be painted over the whole extent.
    A_t = np.exp(-np.where(np.isfinite(tau), tau, 0.0) * geometry_factor)
    R_t = np.full(ntile, np.nan)
    ok = np.isfinite(tiles.sfmc_fcst)
    R_t[ok] = tile_lr_reflectivity(tiles.sfmc_fcst[ok][None, :], tiles.clay[ok], components.incidence_angle_deg,
                                   components.frequency_hz, porosity=tiles.poros[ok])[0]

    # Native-pixel operator, as in build_tile_coefficient_kernel + the tile-opacity-sp tile scaling.
    valid = (K > 0) & (mapping >= 0) & np.isfinite(sigma_factor)
    coeff_pix = np.where(valid, domain_scale * K * sigma_factor * paint(A_t, mapping), np.nan)  # sums to C_t
    contrib_pix = coeff_pix * paint(R_t, mapping)                                                 # sums to H(x)
    C_t = np.bincount(mapping[valid], weights=coeff_pix[valid], minlength=ntile)
    C_t = np.where(C_t > 0, C_t, np.nan)
    H = float(np.nansum(contrib_pix))
    share_t = C_t * R_t / H

    return dict(arm=arm, exp=exp, tiles=tiles, product=product, tv=tv, mapping=mapping, tau=tau, A_t=A_t, R_t=R_t,
                coeff_pix=coeff_pix, contrib_pix=contrib_pix, C_t=C_t, H=H, share_t=share_t)


fields = {arm: arm_fields(arm) for arm in ARMS}

## Checks

1. The recomputed $C_t$ must match the product's coefficients. If it does, the pixel panels are the assimilated
   operator. Expect small per-tile differences: the notebook maps pixels onto the arm's 909-tile domain, while the
   preprocessor mapped them onto the global reference tiles. Their slightly different tile centres move a few hundred
   of about 2 million pixels across tile edges. The weight-weighted difference, and the difference in $H$, should be
   $\lesssim 10^{-3}$.
2. The sum of pixel contributions must equal $\sum_t C_t R_t$.
3. $H(x_f)$ in dB is compared with the observed $y$. This is the raw innovation, before z-score scaling, so it will not
   equal the OFA `innov`.
4. The owner tile. Its raw $C_t$ is 0.80 (the "C = 0.80" in the Discover report); its normalized weight is higher.

In [ ]:
for arm, f in fields.items():
    p = f["product"]
    idx = np.asarray(p["tile_indices0"], dtype=int)
    rel = np.abs(f["C_t"][idx] / p["coefficients"] - 1.0)
    H_tiles = float(np.nansum(p["coefficients"] * f["R_t"][idx]))
    owner = idx[np.argmax(p["coefficient_weights"])]
    print(f"[{arm}] {f['exp']}")
    wrel = float(np.nansum(p["coefficient_weights"] * rel))
    print(f"  support tiles: {idx.size}   C_t recomputed vs product: weight-weighted rel diff {wrel:.2e}   "
          f"max {np.nanmax(rel):.2e} (tile {idx[np.nanargmax(rel)]}, weight "
          f"{p['coefficient_weights'][np.nanargmax(rel)]:.4f})")
    print(f"  H from pixels {f['H']:.6e}   sum_t C_t R_t (product C_t) {H_tiles:.6e}   "
          f"rel diff {abs(f['H'] / H_tiles - 1):.2e}")
    print(f"  H(x_f) = {10*np.log10(f['H']):.3f} dB   observed y = {p['observed_y_db']:.3f} dB   "
          f"raw O-F = {p['observed_y_db'] - 10*np.log10(f['H']):+.3f} dB")
    print(f"  owner tile {owner}: C_t {np.max(p['coefficients']):.3f}  weight {np.max(p['coefficient_weights']):.3f}   "
          f"SRFEXC {f['tv']['srfexc'][owner]:+.3f} kg m-2   dSFMC {f['tv']['sfmc_response'][owner]:+.4f} m3 m-3")
    top = np.argsort(-p["coefficient_weights"])
    print("  tile   weight   share(C_t R_t / H)   tau     A_t    R_t")
    for k in top:
        t = idx[k]
        print(f"  {t:5d}  {p['coefficient_weights'][k]:.4f}   {f['share_t'][t]:.4f}              "
              f"{f['tau'][t]:.4f}  {f['A_t'][t]:.3f}  {f['R_t'][t]:.4f}")

## Figure

In [ ]:
FIGURE_STYLE = {
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "normal", "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "figure.titlesize": 11,
}


def tile_boxes(mapping):
    '''Lon/lat bounding box of each mapped tile's pixels (M36 cells are rectangles at this scale).'''
    half_dx = 0.5 * (dem_extent[1] - dem_extent[0]) / pix_shape[1]
    half_dy = 0.5 * (dem_extent[3] - dem_extent[2]) / pix_shape[0]
    boxes = {}
    for t in np.unique(mapping[mapping >= 0]):
        sel = mapping == t
        boxes[int(t)] = (pix_lon[sel].min() - half_dx, pix_lon[sel].max() + half_dx,
                         pix_lat[sel].min() - half_dy, pix_lat[sel].max() + half_dy)
    return boxes


def draw_tiles(ax, boxes, owner=None, color="k"):
    for t, (w, e, s, n) in boxes.items():
        lw, alpha = (1.6, 1.0) if t == owner else (0.5, 0.55)
        ax.add_patch(Rectangle((w, s), e - w, n - s, fill=False, edgecolor=color, linewidth=lw, alpha=alpha, zorder=4))


def pixel_panel(ax, arr, title, label, cmap, vmin=None, vmax=None, boxes=None, owner=None, box_color="k"):
    im = ax.imshow(np.ma.masked_invalid(arr), extent=dem_extent, origin="upper", cmap=cmap, vmin=vmin, vmax=vmax,
                   interpolation="nearest", aspect="auto", rasterized=True)
    if boxes is not None:
        draw_tiles(ax, boxes, owner, box_color)
    ax.scatter([meta["sp_lon"]], [meta["sp_lat"]], marker="x", s=30, color="k", linewidths=1.0, zorder=6)
    ax.set_xlim(dem_extent[0], dem_extent[1])
    ax.set_ylim(dem_extent[2], dem_extent[3])
    ax.set_title(title)
    ax.set_box_aspect(1)
    ax.xaxis.set_major_locator(MultipleLocator(0.4))
    ax.yaxis.set_major_locator(MultipleLocator(0.2))
    ax.grid(True, alpha=0.15, linewidth=0.5)
    cb = plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02, fraction=0.05)
    cb.set_label(label)
    return im


def log10_top(arr, decades):
    out = np.where(arr > 0, arr, np.nan)
    out = np.log10(out)
    vmax = float(np.nanmax(out))
    return out, vmax - decades, vmax


def symmetric(arr):
    v = float(np.nanmax(np.abs(arr)))
    return -v, v


def plot_story(f, save=True):
    tiles, p, tv, mapping = f["tiles"], f["product"], f["tv"], f["mapping"]
    idx = np.asarray(p["tile_indices0"], dtype=int)
    owner = int(idx[np.argmax(p["coefficient_weights"])])
    boxes = tile_boxes(mapping)
    support_boxes = {t: b for t, b in boxes.items() if t in set(idx.tolist())}
    support_mask = np.isin(mapping, idx)

    with plt.rc_context(FIGURE_STYLE):
        fig, axes = plt.subplots(4, 3, figsize=(11.0, 13.4), constrained_layout=True)
        fig.set_constrained_layout_pads(w_pad=0.03, h_pad=0.03, wspace=0.04, hspace=0.04)
        ax = axes.ravel()

        # --- Row 1: observation ---
        ddm_db = np.vectorize(linear_to_db)(np.where(components.observed_ddm > 0, components.observed_ddm, np.nan))
        im = ax[0].imshow(ddm_db, origin="lower", cmap="viridis", aspect="auto")
        ax[0].add_patch(Rectangle((crop[1].start - 0.5, crop[0].start - 0.5), crop[1].stop - crop[1].start,
                                  crop[0].stop - crop[0].start, fill=False, edgecolor="white", linewidth=1.5))
        ax[0].scatter([center[1]], [center[0]], marker="x", s=35, color="white", linewidths=1.2)
        ax[0].set(title="Observed L1 DDM", xlabel="Doppler bin", ylabel="Delay bin")
        ax[0].set_box_aspect(1)
        plt.colorbar(im, ax=ax[0], shrink=0.85, pad=0.02, fraction=0.05).set_label("Reflectivity (dB)")

        pixel_panel(ax[1], dem, "Copernicus DEM", "Elevation (m)", "terrain")
        k_log, lo, hi = log10_top(K / np.nansum(K), 4)
        pixel_panel(ax[2], k_log, "3×5 crop footprint $K_p$", "log$_{10}$ normalized weight", "magma", lo, hi)

        # --- Row 2: pixel physics ---
        s_log = np.log10(np.where(sigma_factor > 0, sigma_factor, np.nan))
        # Range from the active footprint; a few near-zero pixels would otherwise stretch the scale to -270.
        s_hi = float(np.nanpercentile(s_log[K > 0], 99.5))
        s_lo = s_hi - 6.0
        pixel_panel(ax[3], s_log, "IGOT pixel factor $\\sigma_p$", "log$_{10}$ $\\sigma_{0,base}/R_{base}$", "magma",
                    s_lo, s_hi, boxes, owner, "w")
        pixel_panel(ax[4], paint(f["A_t"], mapping), "Vegetation attenuation $A_t$",
                    "exp(−τ$_t$·2 sec θ) (–)", "YlGn_r", boxes=boxes, owner=owner)
        pixel_panel(ax[5], paint(f["R_t"], mapping), "Forecast reflectivity $R_t$", "$R_t$ (–)", "viridis",
                    boxes=boxes, owner=owner)

        # --- Row 3: pixels -> tiles ---
        c_log, lo, hi = log10_top(f["contrib_pix"] / f["H"], 4)
        pixel_panel(ax[6], c_log, "Pixel contribution to $H(x)$", "log$_{10}$ fraction of $H$", "magma", lo, hi,
                    support_boxes, owner)
        ct_pix = np.where(support_mask, paint(np.log10(f["C_t"]), mapping), np.nan)
        pixel_panel(ax[7], ct_pix, "Tile coefficient $C_t$", "log$_{10}$ $C_t$", "plasma", boxes=support_boxes, owner=owner)
        share_pix = np.where(support_mask, paint(100 * f["share_t"], mapping), np.nan)
        pixel_panel(ax[8], share_pix, "Tile share of $H(x)$: $C_tR_t/H$", "Share (%)", "Blues", 0, 100,
                    boxes=support_boxes, owner=owner)
        for t in idx:
            if 100 * f["share_t"][t] >= 1.0 and t in boxes:
                w, e, s, n = boxes[t]
                share = 100 * f["share_t"][t]
                ax[8].text(0.5 * (w + e), 0.5 * (s + n), f"{share:.0f}%", ha="center", va="center",
                           fontsize=8, color="w" if share > 50 else "k", zorder=7)

        # --- Row 4: DA update ---
        pixel_panel(ax[9], paint(tv["sfmc"], mapping), "Forecast SFMC", "SFMC (m$^3$ m$^{-3}$)", "YlGnBu",
                    boxes=boxes, owner=owner)
        lo, hi = symmetric(tv["srfexc"][np.unique(mapping[mapping >= 0])])
        pixel_panel(ax[10], paint(tv["srfexc"], mapping), "Surface excess increment", "SRFEXC (kg m$^{-2}$)",
                    "RdBu_r", lo, hi, boxes, owner)
        lo, hi = symmetric(tv["sfmc_response"][np.unique(mapping[mapping >= 0])])
        pixel_panel(ax[11], paint(tv["sfmc_response"], mapping), "SFMC response (ana − fcst)",
                    "ΔSFMC (m$^3$ m$^{-3}$)", "RdBu_r", lo, hi, boxes, owner)

        for a, letter in zip(ax, "abcdefghijkl"):
            a.text(0.02, 0.98, f"({letter})", transform=a.transAxes, ha="left", va="top", fontsize=10, zorder=10,
                   bbox=dict(boxstyle="square,pad=0.15", facecolor="white", edgecolor="none", alpha=0.8))
        for a in ax[9:]:
            a.set_xlabel("Longitude (°)")
        for a in ax[[3, 6, 9]]:
            a.set_ylabel("Latitude (°)")

        H_db = 10 * np.log10(f["H"])
        fig.suptitle("\n".join([
            f"CYGNSS L1 operator: cyg{int(meta['sc_num']):02d} sample {meta['sample_id']} ch {meta['ch_id']}, "
            f"{meta['ddm_time'].strftime('%Y-%m-%d %H:%M')} UTC, incidence {components.incidence_angle_deg:.1f}°",
            f"y = {p['observed_y_db']:.2f} dB,  H(x$_f$) = {H_db:.2f} dB  ·  {f['exp']}",
        ]))
        if save:
            stem = OUT_DIR / f"cygl1_operator_story_{CYCLE}_samp{SAMPLE_ID}_ch{CH_ID}_{f['arm']}"
            fig.savefig(stem.with_suffix(".png"), dpi=300)
            fig.savefig(stem.with_suffix(".pdf"), dpi=300)
            print(f"saved {stem}.png/.pdf")
    plt.close(fig)
    return stem.with_suffix(".png") if save else None


# IGOT switches matplotlib to a non-interactive backend, so show the saved PNG.
from IPython.display import Image, display

display(Image(filename=plot_story(fields["err39"]), width=900))

The err2.75 arm uses the same observation and operator ($C_t$ and $A_t$ are identical; $R_t$ differs slightly through
the forecast state). Only the size of the increment changes.

In [ ]:
display(Image(filename=plot_story(fields["err275"]), width=900))

In [ ]:
# Optional: remove the temporary DEM working directory (the persistent Copernicus tiles are untouched).
# import shutil; shutil.rmtree(TEMP_DEM_ROOT, ignore_errors=True)